In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [22]:
print(os.path.expanduser('~'))


/Users/sanmaybandekar


In [23]:
import subprocess
result = subprocess.run(['find', os.path.expanduser('~'), '-name', 'Ecommerce Order Dataset', '-type', 'd'], 
                      capture_output=True, text=True)
print(result.stdout)

/Users/sanmaybandekar/Desktop/ecommerce_project/Ecommerce Order Dataset



In [24]:
os.chdir('/Users/sanmaybandekar/Desktop/ecommerce_project/Ecommerce Order Dataset')

train_orders    = pd.read_csv('train/Orders.csv')
train_items     = pd.read_csv('train/OrderItems.csv')
train_customers = pd.read_csv('train/Customers.csv')
train_payments  = pd.read_csv('train/Payments.csv')
train_products  = pd.read_csv('train/Products.csv')

test_orders    = pd.read_csv('test/Orders.csv')
test_items     = pd.read_csv('test/OrderItems.csv')
test_customers = pd.read_csv('test/Customers.csv')
test_payments  = pd.read_csv('test/Payments.csv')
test_products  = pd.read_csv('test/Products.csv')

# Combine
orders    = pd.concat([train_orders,    test_orders],    ignore_index=True)
items     = pd.concat([train_items,     test_items],     ignore_index=True)
customers = pd.concat([train_customers, test_customers], ignore_index=True)
payments  = pd.concat([train_payments,  test_payments],  ignore_index=True)
products  = pd.concat([train_products,  test_products],  ignore_index=True)

# Verify shapes
for df, name in [(orders,'Orders'),(items,'Items'),
                 (customers,'Customers'),(payments,'Payments'),
                 (products,'Products')]:
    print(f"{name}: {df.shape}")

Orders: (127595, 7)
Items: (127595, 5)
Customers: (127595, 4)
Payments: (127595, 5)
Products: (127595, 6)


In [25]:
print(os.listdir('train'))


['Customers.csv', 'Products.csv', 'Orders.csv', 'OrderItems.csv', 'Payments.csv']


In [48]:
for df, name in [(orders,'Orders'),
                 (items,'Items'),
                 (customers,'Customers'),
                 (payments,'Payments'),
                 (products,'Products')]:
    print(f"\nTable: \n{name}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns:\n{df.columns.tolist()}")
    print(f"\nData Types:\n{df.dtypes}")
    print(f"\nNull Values:\n{df.isnull().sum()}")
    print(f"\nSample:\n{df.head(3)}")


Table: 
Orders
Shape: (127595, 7)

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_timestamp', 'order_estimated_delivery_date']

Data Types:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_timestamp        object
order_estimated_delivery_date    object
dtype: object

Null Values:
order_id                             0
customer_id                          0
order_status                     38279
order_purchase_timestamp             0
order_approved_at                   16
order_delivered_timestamp        40168
order_estimated_delivery_date    38279
dtype: int64

Sample:
       order_id   customer_id order_status order_purchase_timestamp  \
0  Axfy13Hk4PIk  hCT0x9JiGXBQ    delivered      2017-10-22 18:57:54   
1  v6px92oS8cLG  PxA7fv9spyhx    delivered

In [74]:
# Drop rows where order_status is null
orders.dropna(subset=['order_status'], inplace = True)

# Convert all date columns to datetime
date_cols = ['order_purchase_timestamp',
             'order_approved_at',
             'order_delivered_timestamp',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors = 'coerce')

print("Order Status Values: ")
print(orders['order_status'].value_counts())

# Fill missing category with Unknown
products['product_category_name'].fillna('Unknown', inplace=True)

# Fill missing measurements with median
for col in ['product_weight_g','product_length_cm',
            'product_height_cm','product_width_cm']:
    products[col].fillna(products[col].median(), inplace=True)

# CHECK DUPLICATES
for df, name in [(orders,'Orders'),(items,'Items'),
                 (customers,'Customers'),(payments,'Payments'),
                 (products,'Products')]:
    print(f"{name} duplicates: {df.duplicated().sum()}")

print("\nFinal Null Check:")
for df, name in [(orders,'Orders'),(items,'Items'),
                 (customers,'Customers'),(payments,'Payments'),
                 (products,'Products')]:
    print(f"{name}: {df.isnull().sum().sum()} nulls remaining")

Order Status Values: 
order_status
delivered      87428
shipped          936
canceled         409
processing       273
invoiced         266
unavailable        2
approved           2
Name: count, dtype: int64
Orders duplicates: 0
Items duplicates: 0
Customers duplicates: 0
Payments duplicates: 0
Products duplicates: 94644

Final Null Check:
Orders: 1898 nulls remaining
Items: 0 nulls remaining
Customers: 0 nulls remaining
Payments: 0 nulls remaining
Products: 0 nulls remaining


/var/folders/y3/f26tj12s2gz5p9rlc542g9wr0000gn/T/ipykernel_77791/3900894461.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  products[col].fillna(products[col].median(), inplace=True)


In [75]:
# Fix 1 — Products warning (modern pandas way)
products['product_category_name'] = products['product_category_name'].fillna('Unknown')

for col in ['product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].median())

# Fix 2 — Drop product duplicates (keep first occurrence)
products.drop_duplicates(inplace=True)

# Fix 3 — Orders nulls are okay — only delivered orders have delivery timestamp
# Let's just verify
print("Orders nulls remaining:")
print(orders.isnull().sum())

print(f"\nProducts shape after removing duplicates: {products.shape}")
print(f"Products duplicates remaining: {products.duplicated().sum()}")

Orders nulls remaining:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
dtype: int64

Products shape after removing duplicates: (32951, 6)
Products duplicates remaining: 0


In [76]:
# Merge all tables step by step
ecommerce_master = orders.merge(customers, on='customer_id', how='left')
ecommerce_master = ecommerce_master.merge(items, on='order_id', how='left')
ecommerce_master = ecommerce_master.merge(products, on='product_id', how='left')
ecommerce_master = ecommerce_master.merge(payments, on='order_id', how='left')

# Check result
print(f"Master table shape: {ecommerce_master.shape}")
print(f"\nColumns: {ecommerce_master.columns.tolist()}")
print(f"\nNull check:\n{ecommerce_master.isnull().sum()}")
print(f"\nSample:\n{ecommerce_master.head(3)}")

Master table shape: (89316, 23)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_timestamp', 'order_estimated_delivery_date', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_id', 'seller_id', 'price', 'shipping_charges', 'product_category_name', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Null check:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   9
order_delivered_timestamp        1889
order_estimated_delivery_date       0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
product_id                          0
seller_id                           0
price                        

In [77]:
# Export master table
ecommerce_master.to_csv('ecommerce_master.csv', index=False)
print("Master table saved!")

Master table saved!


In [78]:
print("Total Revenue:", round(ecommerce_master['price'].sum(), 2))
print("Total Orders:", ecommerce_master['order_id'].nunique())
print("Total Customers:", ecommerce_master['customer_id'].nunique())
print("Avg Order Value:", round(ecommerce_master.groupby('order_id')['price'].sum().mean(), 2))

Total Revenue: 30447872.89
Total Orders: 89316
Total Customers: 89316
Avg Order Value: 340.9


In [79]:
ecommerce_master['month'] = ecommerce_master['order_purchase_timestamp'].dt.to_period('M')
monthly_revenue = ecommerce_master.groupby('month')['price'].sum().reset_index()
monthly_revenue.columns = ['month', 'revenue']
print("\nMonthly Revenue:")
print(monthly_revenue)


Monthly Revenue:
      month     revenue
0   2016-09      944.56
1   2016-10    96038.16
2   2016-12      215.94
3   2017-01   242128.17
4   2017-02   446158.45
5   2017-03   826478.58
6   2017-04   736335.08
7   2017-05  1028211.53
8   2017-06   915599.54
9   2017-07  1217621.46
10  2017-08  1271253.67
11  2017-09  1432583.39
12  2017-10  1430263.65
13  2017-11  2848052.62
14  2017-12  1792221.66
15  2018-01  2035920.92
16  2018-02  2004865.64
17  2018-03  2340055.36
18  2018-04  2199653.24
19  2018-05  2388603.88
20  2018-06  1734187.77
21  2018-07  1761504.83
22  2018-08  1698819.39
23  2018-09      155.40


In [80]:
top_categories = ecommerce_master.groupby('product_category_name')['price'].sum().nlargest(10).reset_index()
top_categories.columns = ['category', 'revenue']
print("\nTop 10 Categories:")
print(top_categories)


Top 10 Categories:
                category      revenue
0                   toys  22994700.31
1        furniture_decor    753073.49
2           garden_tools    746590.63
3         bed_bath_table    604534.89
4          health_beauty    596724.24
5          watches_gifts    584225.46
6         sports_leisure    547922.16
7              telephony    494999.59
8  computers_accessories    430532.98
9             housewares    415090.22


In [81]:
payment_split = ecommerce_master['payment_type'].value_counts()
print("\nPayment Types:")
print(payment_split)


Payment Types:
payment_type
credit_card    65814
wallet         17302
voucher         4911
debit_card      1289
Name: count, dtype: int64


In [82]:
status_split = ecommerce_master['order_status'].value_counts()
print("\nOrder Status:")
print(status_split)


Order Status:
order_status
delivered      87428
shipped          936
canceled         409
processing       273
invoiced         266
unavailable        2
approved           2
Name: count, dtype: int64


In [83]:
pip install sqlalchemy mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 14.5 MB/s  0:00:015.7 MB/s eta 0:00:01:01
Note: you may need to restart the kernel to use updated packages.


In [84]:
import urllib.parse
from sqlalchemy import create_engine

username = "root"
password = urllib.parse.quote_plus("StrongPass@123")  #enter your password here
host = "127.0.0.1"
port = "3306"
database = "ecommerce_db"

engine = create_engine(f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}")

# Test connection first
try:
    with engine.connect() as conn:
        print("Connected successfully!")
except Exception as e:
    print(f"Connection failed: {e}")

Connected successfully!


In [86]:
# Fix month column - convert Period to string
ecommerce_master['month'] = ecommerce_master['month'].astype(str)

# Now load all tables
tables = {
    'orders': orders,
    'items': items,
    'customers': customers,
    'payments': payments,
    'products': products,
    'ecommerce_master': ecommerce_master
}

for table_name, df in tables.items():
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"Loaded: {table_name}")

print("All tables loaded into MySQL!")

Loaded: orders
Loaded: items
Loaded: customers
Loaded: payments
Loaded: products
Loaded: ecommerce_master
All tables loaded into MySQL!


In [88]:
# Note: Data cleaning and SQL analysis performed on macOS (Jupyter Notebook + MySQL Workbench)
# CSV files exported here for Power BI dashboard development on Windows

# Export all clean tables

orders.to_csv('orders_clean.csv', index=False)
items.to_csv('items_clean.csv', index=False)
customers.to_csv('customers_clean.csv', index=False)
payments.to_csv('payments_clean.csv', index=False)
products.to_csv('products_clean.csv', index=False)
ecommerce_master.to_csv('ecommerce_master.csv', index=False)

print("All files exported!")

All files exported!
